# 07 - Sistemas não lineares
Vamos aprender sobre como usar o método iterativo de Newton-Raphson adaptado aos sistemas de equações, envolvendo aproximações iniciais, cálculo da matriz do Jacobiano e critério de parada, para resolução de sistemas não lineares.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana7
```

Faça o checkout nessa nova branch:

```bash
git checkout semana7
```

<hr />

## Atividade 1
Escreva o método iterativo de Newton-Raphson adaptado dentro do arquivo $utils/algoritmos.py$.

In [ ]:
import numpy as np
from scipy.optimize import approx_fprime


def JN(x, F, eps=1e-8):
    x = np.asarray(x, dtype=float)
    n = x.size
    Jnum = np.zeros((n, n), dtype=float)

    def fi(v):
        return F(v)[i]

    for i in range(n):
        Jnum[i, :] = approx_fprime(x, fi, epsilon=eps)

    return Jnum


def G(x, F, J):
    return x - np.linalg.inv(J(x)) @ F(x)


def GN(x, F):
    return x - np.linalg.inv(JN(x, F)) @ F(x)


def fixed_point(a, g, TOL=1e-8, iter=1000):
    x = g(a)
    i = 1
    while np.linalg.norm(x - a) > TOL and i < iter:
        a = x
        x = g(a)
        i += 1
    return x

Encontre uma aproximação numérica para o seguinte problema não linear de três equações e três incógnitas:

$$
2x_{1} - x_{2} = \cos(x_{1})
$$

$$
- x_{1} + 2x_{2} - x_{3} = \cos(x_{2})
$$

$$
- x_{2} + x_{3} = \cos(x_{3})
$$

Partindo da seguinte aproximação inicial:

$$
x^{(0)} = (1,\,1,\,1)^{T}
$$

In [2]:
import sys
import os
import numpy as np

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import (
    G,
    GN,
    fixed_point,
)


def main():

    def F(x):
        x1, x2, x3 = x
        return np.array(
            [
                2.0 * x1 - x2 - np.cos(x1),
                -x1 + 2.0 * x2 - x3 - np.cos(x2),
                -x2 + x3 - np.cos(x3),
            ],
            dtype=float,
        )

    def J(x):
        x1, x2, x3 = x
        return np.array(
            [
                [2.0 + np.sin(x1), -1.0, 0.0],
                [-1.0, 2.0 + np.sin(x2), -1.0],
                [0.0, -1.0, 1.0 + np.sin(x3)],
            ],
            dtype=float,
        )

    x = np.array([1.0, 1.0, 1.0], dtype=float)
    r = fixed_point(x, lambda x: G(x, F, J))
    print(r)
    r = fixed_point(x, lambda x: GN(x, F))
    print(r)


if __name__ == "__main__":
    main()

[0.95127255 1.32189759 1.44618584]
[0.95127255 1.32189759 1.44618584]


## Atividade 2
Encontre uma aproximação em cada incógnita para a solução próxima da origem do sistema:

$$
6x - 2y + e^{z} = 2
$$

$$
\sin(x) - y + z = 0
$$

$$
\sin(x) + 2y + 3z = 1
$$

**Resposta esperada:**

$$
x \approx 0.259751, \quad y \approx 0.302736, \quad z \approx 0.045896
$$

In [ ]:
import numpy as np


def F(v):
    x, y, z = v
    return np.array([
        6*x - 2*y + np.exp(z) - 2,
        np.sin(x) - y + z,
        np.sin(x) + 2*y + 3*z - 1,
    ])


def jacobiano(v):
    x, _, z = v
    return np.array([
        [6.0, -2.0, np.exp(z)],
        [np.cos(x), -1.0, 1.0],
        [np.cos(x), 2.0, 3.0],
    ])


v = np.zeros(3)
tolerancia = 1e-10
print(f"{'k':>2} {'x':>14} {'y':>14} {'z':>14} {'||F(v)||':>14}")
print(f"{0:2d} {v[0]:14.8f} {v[1]:14.8f} {v[2]:14.8f} {np.linalg.norm(F(v)):14.6e}")

for k in range(1, 101):
    passo = np.linalg.solve(jacobiano(v), -F(v))
    v = v + passo
    residuo = np.linalg.norm(F(v))
    print(f"{k:2d} {v[0]:14.8f} {v[1]:14.8f} {v[2]:14.8f} {residuo:14.6e}")
    if np.linalg.norm(passo) < tolerancia:
        break
else:
    raise RuntimeError("Newton-Raphson nao convergiu em 100 iteracoes.")

print(f"\\nSolucao aproximada: x={v[0]:.6f}, y={v[1]:.6f}, z={v[2]:.6f}")

## Atividade 3
Considere o problema de encontrar os pontos de interseção das curvas descritas:

$$
\frac{x^{2}}{8} + \frac{(y-1)^{2}}{5} = 1
$$

$$
\tan^{-1}(x) + x = y + y^{3}
$$

Com base no gráfico, encontre soluções aproximadas para o problema e use-as para iniciar o método de Newton-Raphson para encontrar as raízes.

**Resposta esperada:**

$$
(-1.2085435,\,-1.0216674) 
\quad \text{e} \quad 
(2.7871115,\,1.3807962)
$$

In [ ]:
import numpy as np


def F(v):
    x, y = v
    return np.array([
        x**2 / 8 + (y - 1)**2 / 5 - 1,
        np.arctan(x) + x - y - y**3,
    ])


def jacobiano(v):
    x, y = v
    return np.array([
        [x / 4, 2 * (y - 1) / 5],
        [1 / (1 + x**2) + 1, -1 - 3 * y**2],
    ])


def newton_sistema(chute_inicial, tolerancia=1e-12, max_iteracoes=100):
    v = np.array(chute_inicial, dtype=float)
    print(f"Chute inicial: ({v[0]:.6f}, {v[1]:.6f})")
    for k in range(1, max_iteracoes + 1):
        passo = np.linalg.solve(jacobiano(v), -F(v))
        v = v + passo
        residuo = np.linalg.norm(F(v))
        print(f"k={k}: x={v[0]:.10f}, y={v[1]:.10f}, ||F||={residuo:.3e}")
        if np.linalg.norm(passo) < tolerancia and residuo < tolerancia:
            return v
    raise RuntimeError("Newton-Raphson nao convergiu.")


raiz_1 = newton_sistema([-1.2, -1.0])
print(f"\\nPrimeira intersecao: ({raiz_1[0]:.7f}, {raiz_1[1]:.7f})\\n")

raiz_2 = newton_sistema([2.8, 1.4])
print(f"\\nSegunda intersecao: ({raiz_2[0]:.7f}, {raiz_2[1]:.7f})")

## Atividade 4
Uma indústria consome energia elétrica de três usinas fornecedoras.  

O custo de fornecimento em reais por hora como função da potência consumida em kW é dada pelas seguintes funções:

$$
C_{1}(x) = 10 + 0.3x + 10^{-4}x^{2} + 3.4 \cdot 10^{-9}x^{4} \tag{5.61}
$$

$$
C_{2}(x) = 50 + 0.25x + 2 \cdot 10^{-4}x^{2} + 4.3 \cdot 10^{-7}x^{3}
$$

$$
C_{3}(x) = 500 + 0.19x + 5 \cdot 10^{-4}x^{2} + 1.1 \cdot 10^{-7}x^{4}
$$

Calcule a distribuição de consumo que produz custo mínimo quando a potência total consumida é **1500 kW**.  

Denote por $x_{1}, x_{2}, x_{3}$ as potências consumidas das usinas 1, 2 e 3, respectivamente.  

O custo total será dado por:

$$
C(x_{1},x_{2},x_{3}) = C_{1}(x_{1}) + C_{2}(x_{2}) + C_{3}(x_{3})
$$

Enquanto o consumo total é:

$$
G(x_{1},x_{2},x_{3}) = x_{1} + x_{2} + x_{3} - 1500 = 0
$$

Pelos multiplicadores de Lagrange, temos que resolver o sistema:

$$
\nabla C(x_{1},x_{2},x_{3}) = \lambda \nabla G(x_{1},x_{2},x_{3})
$$

$$
G(x_{1},x_{2},x_{3}) = 0
$$

**Resposta esperada:**

$$
(x_{1},x_{2},x_{3}) \approx (453.62, \; 901.94, \; 144.43)
$$

In [ ]:
import numpy as np

potencia_total = 1500.0


def gradientes_custo(x1, x2, x3):
    return np.array([
        0.3 + 2e-4*x1 + 1.36e-8*x1**3,
        0.25 + 4e-4*x2 + 1.29e-6*x2**2,
        0.19 + 1e-3*x3 + 4.4e-7*x3**3,
    ])


def F(v):
    x1, x2, x3, lamb = v
    return np.append(gradientes_custo(x1, x2, x3) - lamb, x1 + x2 + x3 - potencia_total)


def jacobiano(v):
    x1, x2, x3, _ = v
    return np.array([
        [2e-4 + 4.08e-8*x1**2, 0.0, 0.0, -1.0],
        [0.0, 4e-4 + 2.58e-6*x2, 0.0, -1.0],
        [0.0, 0.0, 1e-3 + 1.32e-6*x3**2, -1.0],
        [1.0, 1.0, 1.0, 0.0],
    ])


v = np.array([453.62, 901.94, 144.43, 1.66])
tolerancia = 1e-11

for k in range(1, 21):
    passo = np.linalg.solve(jacobiano(v), -F(v))
    v += passo
    if np.linalg.norm(passo) < tolerancia and np.linalg.norm(F(v)) < tolerancia:
        break
else:
    raise RuntimeError("Newton-Raphson nao convergiu.")

x1, x2, x3, lamb = v

def custo1(x):
    return 10 + 0.3*x + 1e-4*x**2 + 3.4e-9*x**4

def custo2(x):
    return 50 + 0.25*x + 2e-4*x**2 + 4.3e-7*x**3

def custo3(x):
    return 500 + 0.19*x + 5e-4*x**2 + 1.1e-7*x**4

custo_total = custo1(x1) + custo2(x2) + custo3(x3)

print(f"Iteracoes de Newton: {k}")
print(f"Potencias: x1={x1:.8f} kW, x2={x2:.8f} kW, x3={x3:.8f} kW")
print(f"Com quatro algarismos significativos: ({x1:.4g}, {x2:.4g}, {x3:.4g}) kW")
print(f"Multiplicador lambda: {lamb:.10f}")
print(f"Potencia total: {x1 + x2 + x3:.10f} kW")
print(f"Custo total: R$ {custo_total:.6f}/h")
print(f"Norma do residuo: {np.linalg.norm(F(v)):.3e}")

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 7"
git push origin semana7
```